# Text-to-SQL with Vanna.ai - Complete Tutorial

**Duration:** 1-2 hours  
**Level:** Intermediate (requires SQL knowledge and basic LLM understanding)  
**Framework:** Vanna.ai 2.0 Agent Framework  
**Database:** PostgreSQL (Supabase)

---

## What You'll Learn

1. What Text-to-SQL is and why it matters
2. How to set up Vanna.ai 2.0 with OpenAI and PostgreSQL
3. Understanding the Agent framework architecture
4. Generating SQL from natural language questions
5. Executing queries and handling results
6. Best practices for production deployment

---

## Prerequisites

- OpenAI API key
- PostgreSQL database (Supabase or local)
- Python 3.12+
- Basic understanding of SQL and LLMs

---

# Section 1: Introduction to Text-to-SQL

## What is Text-to-SQL?

Text-to-SQL converts natural language questions into SQL queries:

```
Question: "How many customers do we have?"
         ↓
SQL: SELECT COUNT(*) FROM customers
         ↓
Result: 100
```

## Why Use Text-to-SQL?

1. **Democratize Data Access** - Non-technical users can query databases
2. **Faster Analytics** - No manual SQL writing required
3. **Business Intelligence** - Power chatbots and dashboards
4. **Real-World Use Cases**: Slack bots, analytics dashboards, customer support tools

## Technical Architecture

```
Natural Language Question
         ↓
    LLM (OpenAI GPT-4o)
         ↓
    SQL Generation
         ↓
    PostgreSQL Database
         ↓
    Results (DataFrame)
```

## Vanna.ai 2.0 Overview

- **Agent Framework** - Modern architecture with user awareness
- **Modular Design** - Swap LLMs, databases, tools
- **Security** - User permissions and access control
- **Streaming UI** - Real-time component updates

---

# Section 2: Environment Setup

## Install Dependencies

In [1]:
# Install required packages
#!pip install vanna openai psycopg2-binary python-dotenv pandas sqlalchemy -q

## Import Libraries

**Important:** Vanna 2.0 uses different imports than Legacy (0.x)

In [2]:
# Core Vanna 2.0 imports
from vanna import Agent
from vanna.integrations.openai import OpenAILlmService
from vanna.integrations.postgres import PostgresRunner
from vanna.core.registry import ToolRegistry
from vanna.tools import RunSqlTool
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.local.agent_memory import DemoAgentMemory

# Standard libraries
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from urllib.parse import urlparse
from IPython.display import display

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Load Environment Variables

In [3]:
# Load from .env file
load_dotenv()

# Get credentials
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")

# Verify credentials exist
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment")
if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment")

print("✓ Environment variables loaded")
print(f"  OpenAI API Key: {OPENAI_API_KEY[:20]}...")
print(f"  Database: {DATABASE_URL.split('@')[1] if '@' in DATABASE_URL else 'configured'}")

✓ Environment variables loaded
  OpenAI API Key: sk-proj-ctk5Lu3hTJwG...
  Database: aws-1-us-east-2.pooler.supabase.com:5432/postgres


---

# Section 3: Initialize Vanna 2.0 Agent

## Architecture Components

Vanna 2.0 uses a modular Agent architecture:

1. **LlmService** - The language model (OpenAI GPT-4o)
2. **SqlRunner** - Database connection and execution
3. **ToolRegistry** - Available tools for the Agent
4. **UserResolver** - User authentication and permissions
5. **AgentMemory** - Conversation history
6. **Agent** - Orchestrates everything

## Step 1: Initialize LLM Service

In [4]:
# Initialize OpenAI GPT-4o
llm = OpenAILlmService(
    api_key=OPENAI_API_KEY,
    model="gpt-4o"  # Latest OpenAI model
)

print("✓ LLM Service initialized (OpenAI GPT-4o)")

✓ LLM Service initialized (OpenAI GPT-4o)


## Step 2: Initialize PostgreSQL Runner

**Important:** Use `connection_string` approach (simplest)

In [5]:
# Connect to PostgreSQL using connection string
postgres_runner = PostgresRunner(
    connection_string=DATABASE_URL
)

print("✓ PostgreSQL Runner initialized")

✓ PostgreSQL Runner initialized


## Step 3: Register Tools

Tools define what the Agent can do. We'll register the `RunSqlTool`.

In [6]:
# Create tool registry
tools = ToolRegistry()

# Register SQL execution tool
tools.register_local_tool(
    RunSqlTool(sql_runner=postgres_runner),
    access_groups=['user', 'admin']  # Who can use this tool
)

print("✓ Tools registered (RunSqlTool)")

✓ Tools registered (RunSqlTool)


## Step 4: Create User Resolver

User resolver handles authentication and permissions.

In [7]:
class SimpleUserResolver(UserResolver):
    """Simple user resolver for tutorial purposes"""
    
    async def resolve_user(self, request_context: RequestContext) -> User:
        return User(
            id="tutorial_user",
            email="student@tutorial.com",
            group_memberships=['user', 'admin']  # Full access
        )

user_resolver = SimpleUserResolver()

print("✓ User Resolver created")

✓ User Resolver created


## Step 5: Initialize Agent

The Agent orchestrates all components.

In [8]:
# Create the Agent
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=user_resolver,
    agent_memory=DemoAgentMemory()  # In-memory conversation history
)

print("✓ Agent initialized successfully!")
print("\n=== Vanna 2.0 Agent Ready ===")

✓ Agent initialized successfully!

=== Vanna 2.0 Agent Ready ===


---

# Section 4: Understanding the Database

## Database Schema Overview

Our e-commerce database has 3 tables:

### 1. `customers` (100 rows)
- `id` - Primary key
- `name` - Customer name
- `email` - Email address
- `segment` - SMB, Enterprise, or Individual
- `country` - Customer country

### 2. `products` (50 rows)
- `id` - Primary key
- `name` - Product name
- `category` - Product category
- `price` - Product price
- `stock_quantity` - Inventory count

### 3. `orders` (200 rows)
- `id` - Primary key
- `customer_id` - Foreign key to customers
- `order_date` - Order date
- `total_amount` - Order total (NOT 'price'!)
- `status` - Pending, Delivered, Cancelled, Processing

### Relationships
- `customers.id` → `orders.customer_id` (one-to-many)

## Create Helper Function for Direct SQL

Sometimes we want to run SQL directly without the Agent.

In [9]:
def run_sql_simple(sql: str) -> pd.DataFrame:
    """
    Execute SQL directly using psycopg2 (non-async, simple).
    
    Args:
        sql: SQL query string
        
    Returns:
        DataFrame with results
    """
    # Parse DATABASE_URL
    parsed = urlparse(DATABASE_URL)
    
    # Connect to database
    conn = psycopg2.connect(
        host=parsed.hostname,
        database=parsed.path[1:],  # Remove leading '/'
        user=parsed.username,
        password=parsed.password,
        port=parsed.port or 5432
    )
    
    # Execute query
    df = pd.read_sql_query(sql, conn)
    conn.close()
    
    return df

print("✓ Helper function created: run_sql_simple()")

✓ Helper function created: run_sql_simple()


## Preview Database Tables

In [10]:
# Preview customers table
print("=== CUSTOMERS TABLE (Sample) ===")
customers_sample = run_sql_simple("SELECT * FROM customers LIMIT 5")
display(customers_sample)

=== CUSTOMERS TABLE (Sample) ===


/var/folders/xz/5_crg54x7fqbg35gh1v4rcbh0000gn/T/ipykernel_32684/4022728488.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,id,name,email,segment,country,created_at,updated_at
0,1,James Morales,bturner@example.net,SMB,India,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
1,2,Jose Hayes,perezkimberly@example.com,SMB,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
2,3,Anna Evans,annawoodward@example.com,SMB,Germany,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
3,4,Gina Yang MD,jacqueline62@example.com,Enterprise,India,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
4,5,Charles Wilson,karencarter@example.com,Individual,Singapore,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798


In [11]:
# Preview products table
print("=== PRODUCTS TABLE (Sample) ===")
products_sample = run_sql_simple("SELECT * FROM products LIMIT 5")
display(products_sample)

=== PRODUCTS TABLE (Sample) ===


/var/folders/xz/5_crg54x7fqbg35gh1v4rcbh0000gn/T/ipykernel_32684/4022728488.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,id,name,category,price,stock_quantity,description,created_at,updated_at
0,1,Case,Accessories,1720.09,59,Mention western while tell sell involve. Quick...,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
1,2,Case Henderson-Wright,Accessories,938.55,188,None,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
2,3,Mount Cooper-Ray,Accessories,900.03,402,To especially military continue entire such. T...,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
3,4,"Analytics Platform Jones, Wolfe and Sanchez",Software,1958.66,395,Fish by character conference more. Mention bil...,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
4,5,Charger Pollard-Williams,Accessories,1805.79,113,Fill mean news young nation operation. Televis...,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798


In [12]:
# Preview orders table
print("=== ORDERS TABLE (Sample) ===")
orders_sample = run_sql_simple("SELECT * FROM orders LIMIT 5")
display(orders_sample)

=== ORDERS TABLE (Sample) ===


/var/folders/xz/5_crg54x7fqbg35gh1v4rcbh0000gn/T/ipykernel_32684/4022728488.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,id,customer_id,order_date,total_amount,status,shipping_address,created_at,updated_at
0,1,56,2026-01-18,1152.63,Cancelled,None,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
1,2,91,2025-06-09,83.31,Delivered,"71392 Kelly Well\nFitzgeraldside, NM 77501",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
2,3,91,2025-12-18,2349.80,Delivered,Unit 5594 Box 6584\nDPO AA 94613,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
3,4,74,2026-03-21,3058.13,Delivered,"3460 Fischer Flat\nNorth Christopher, WI 49313",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
4,5,37,2026-02-15,1646.48,Delivered,"7750 Cathy Cape\nEast Mark, KS 08991",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798


## Database Statistics

In [13]:
# Get table counts
stats = run_sql_simple("""
SELECT 
    (SELECT COUNT(*) FROM customers) as total_customers,
    (SELECT COUNT(*) FROM products) as total_products,
    (SELECT COUNT(*) FROM orders) as total_orders,
    (SELECT COUNT(DISTINCT segment) FROM customers) as customer_segments,
    (SELECT COUNT(DISTINCT category) FROM products) as product_categories,
    (SELECT COUNT(DISTINCT status) FROM orders) as order_statuses
""")

print("=== DATABASE STATISTICS ===")
display(stats)

/var/folders/xz/5_crg54x7fqbg35gh1v4rcbh0000gn/T/ipykernel_32684/4022728488.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


=== DATABASE STATISTICS ===


,total_customers,total_products,total_orders,customer_segments,product_categories,order_statuses
0,100,50,200,3,6,4


---

# Section 5: Provide Schema Documentation to Agent

## Why Schema Documentation Matters

The Agent needs to understand:
- What tables exist
- What columns each table has
- Data types and relationships
- Business terminology

**Without schema knowledge, the Agent will generate incorrect SQL!**

Example: It might look for a `price` column in `orders` when the correct column is `total_amount`.

## Provide Schema Context

We'll give the Agent a detailed schema description.

In [14]:
# Schema documentation
SCHEMA_CONTEXT = """
DATABASE SCHEMA:

Table: customers
Columns:
  - id (SERIAL PRIMARY KEY)
  - name (VARCHAR) - Customer full name
  - email (VARCHAR) - Customer email address
  - segment (VARCHAR) - One of: 'SMB', 'Enterprise', 'Individual'
  - country (VARCHAR) - Customer country
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

Table: products
Columns:
  - id (SERIAL PRIMARY KEY)
  - name (VARCHAR) - Product name
  - category (VARCHAR) - Product category (Electronics, Software, Hardware, etc.)
  - price (DECIMAL) - Product unit price
  - stock_quantity (INT) - Current inventory count
  - description (TEXT)
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

Table: orders
Columns:
  - id (SERIAL PRIMARY KEY)
  - customer_id (INT) - Foreign key to customers.id
  - order_date (DATE) - Date of order
  - total_amount (DECIMAL) - TOTAL ORDER PRICE (use this for revenue, NOT 'price'!)
  - status (VARCHAR) - One of: 'Pending', 'Delivered', 'Cancelled', 'Processing'
  - shipping_address (TEXT)
  - created_at (TIMESTAMP)
  - updated_at (TIMESTAMP)

RELATIONSHIPS:
  - customers.id → orders.customer_id (one-to-many)
  
IMPORTANT NOTES:
  - For order revenue/pricing, use orders.total_amount (NOT 'price')
  - Customer segments: 'SMB', 'Enterprise', 'Individual' (case-sensitive)
  - Order statuses: 'Pending', 'Delivered', 'Cancelled', 'Processing' (case-sensitive)
  - To join customers and orders: JOIN orders ON customers.id = orders.customer_id
"""

print("✓ Schema context prepared")
print("\nThis will help the Agent understand the database structure.")

✓ Schema context prepared

This will help the Agent understand the database structure.


---

# Section 6: Querying with the Agent

## Create Helper Function for Agent Queries

**Important:** Vanna 2.0's `agent.send_message()` returns an **async generator**, not a simple awaitable.

We must use `async for` to iterate through UI components.

In [15]:
# Create a request context (simulates HTTP request)
request_context = RequestContext()

# Test query
print("🧪 Testing Agent with: 'How many customers are in the database?'\n")

try:
    # Iterate over streaming results
    async for ui_component in agent.send_message(
        request_context=request_context,
        message="How many customers are in the database?"
    ):
        # Each ui_component is a piece of the response
          print(ui_component)

    print("\n✅ Agent responded successfully!")

except Exception as e:
      print(f"❌ Error: {e}")
      print("\nTroubleshooting:")
      print("  • Make sure DATABASE_URL is set correctly")
      print("  • Ensure the database has a 'customers' table")
      print("  • Check OPENAI_API_KEY is valid")

🧪 Testing Agent with: 'How many customers are in the database?'

timestamp='2026-04-28T05:08:58.523946' rich_component=StatusBarUpdateComponent(id='vanna-status-bar', type=<ComponentType.STATUS_BAR_UPDATE: 'status_bar_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2026-04-28T05:08:58.523928', visible=True, interactive=False, status='working', message='Processing your request...', detail='Analyzing query') simple_component=None
timestamp='2026-04-28T05:08:58.524325' rich_component=TaskTrackerUpdateComponent(id='vanna-task-tracker', type=<ComponentType.TASK_TRACKER_UPDATE: 'task_tracker_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2026-04-28T05:08:58.524091', visible=True, interactive=False, operation=<TaskOperation.ADD_TASK: 'add_task'>, task=Task(id='1071e084-3718-4f1f-81fa-70e8033e3ca5', title='Load conversation context', description='Reading message history and user context', status='pending',

In [16]:
async def ask_agent(question: str, include_schema: bool = True):
    """
    Ask the Agent a question and display results.
    
    Args:
        question: Natural language question
        include_schema: Whether to include schema context (recommended: True)
    """
    # Prepend schema context to question
    if include_schema:
        full_message = f"{SCHEMA_CONTEXT}\n\nQUESTION: {question}"
    else:
        full_message = question
    
    # Create request context
    request_context = RequestContext()
    
    print(f"🤔 Question: {question}\n")
    
    # Send message to agent (returns async generator)
    async for component in agent.send_message(
        request_context=request_context,
        message=full_message
    ):
        # Extract the UI component
        rich_comp = component.rich_component
        
        # Handle different component types
        if hasattr(rich_comp, 'rows') and rich_comp.rows:
            # DataFrameComponent - extract data from 'rows' attribute
            df = pd.DataFrame(rich_comp.rows)
            print("📊 Results:")
            display(df)
            print()
        
        elif hasattr(rich_comp, 'text') and rich_comp.text:
            # RichTextComponent - display text
            print(f"💬 {rich_comp.text}\n")
        
        elif hasattr(rich_comp, 'sql') and rich_comp.sql:
            # SQL code component
            print(f"🔍 Generated SQL:\n{rich_comp.sql}\n")

print("✓ Helper function created: ask_agent()")

✓ Helper function created: ask_agent()


### With Explanations

In [17]:
# async def ask_agent(question: str):
#       """Ask agent and display results properly."""
#       print(f"❓ Question: {question}\n")

#       request_context = RequestContext()

#       print("="*80 + "\n")

#       async for component in agent.send_message(
#           request_context=request_context,
#           message=question
#       ):
#           rich_comp = component.rich_component

#           # Check for DataFrame component
#           if hasattr(rich_comp, 'type') and 'DATAFRAME' in str(rich_comp.type):
#               # ✅ Get data from 'rows' attribute!
#               if hasattr(rich_comp, 'rows') and rich_comp.rows:
#                   df = pd.DataFrame(rich_comp.rows)

#                   print("📊 Query Results:\n")
#                   from IPython.display import display
#                   display(df)
#                   print()

#           # Check for text explanation
#           elif hasattr(rich_comp, 'type') and 'TEXT' in str(rich_comp.type):
#               if hasattr(rich_comp, 'content') and rich_comp.content:
#                   print("💬 Explanation:\n")
#                   print(rich_comp.content)
#                   print()

#       print("="*80 + "\n")

## Example 1: Simple COUNT Query

In [18]:
await ask_agent("How many customers do we have?")

🤔 Question: How many customers do we have?

📊 Results:


,customer_count
0,100


## Example 2: Aggregation Query

In [19]:
await ask_agent("What is the total revenue from all orders?")

🤔 Question: What is the total revenue from all orders?

📊 Results:


,total_revenue
0,500684.33


## Example 3: Filtering Query

In [20]:
await ask_agent("How many orders have been delivered?")

🤔 Question: How many orders have been delivered?

📊 Results:


,count
0,133


## Example 4: JOIN Query

In [21]:
await ask_agent("Show me the top 5 customers by total spending")

🤔 Question: Show me the top 5 customers by total spending

📊 Results:


,id,name,total_spending
0,53,April Wilson,15784.48
1,27,David Tapia,14847.71
2,74,Larry Cook,14362.40
3,4,Gina Yang MD,13913.67
4,97,Johnathan Mitchell,13527.85


## Example 5: GROUP BY with JOIN

In [22]:
await ask_agent("What is the average order value for each customer segment?")

🤔 Question: What is the average order value for each customer segment?

📊 Results:


,segment,average_order_value
0,Individual,2463.5437096774193548
1,Enterprise,2549.0903389830508475
2,SMB,2500.6112658227848101


## Example 6: Time-Based Filtering

In [23]:
await ask_agent("Show orders from the last 30 days")

🤔 Question: Show orders from the last 30 days

📊 Results:


,id,customer_id,order_date,total_amount,status,shipping_address,created_at,updated_at
0,6,16,2026-03-30,3105.04,Processing,"389 Jennifer Ville Suite 641\nSouth Roberto, I...",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
1,16,5,2026-04-10,2583.72,Pending,"0978 Lisa Expressway\nSouth Edwinhaven, AL 95774",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
2,52,14,2026-04-14,441.43,Delivered,"403 Amber Key\nDanielleton, TN 76275",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
3,54,70,2026-04-06,4988.40,Cancelled,None,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
4,65,52,2026-04-10,790.10,Delivered,"7776 Miller Vista Apt. 982\nDarrenmouth, GA 94787",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
5,69,66,2026-04-20,3691.51,Pending,"400 Mclaughlin Pines\nEast Kevinchester, VA 14920",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
6,86,16,2026-04-17,2194.61,Delivered,None,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
7,90,35,2026-03-30,1089.05,Delivered,"597 Monica Road Apt. 209\nWilliamborough, IL 6...",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
8,91,14,2026-04-02,965.36,Delivered,"388 Johnson Canyon Suite 951\nKristinborough, ...",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
9,95,4,2026-04-04,3131.72,Delivered,"673 James Parkway\nSouth Julian, AL 23967",2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798


## Example 7: Complex Business Question

In [24]:
await ask_agent("What is the total revenue from delivered orders placed by Enterprise customers?")

🤔 Question: What is the total revenue from delivered orders placed by Enterprise customers?

📊 Results:


,total_revenue
0,78133.57


---

# Section 7: Understanding How the Agent Works

## The Agent's Workflow

When you ask a question, here's what happens:

1. **Question Processing** - Agent receives your natural language question
2. **Schema Retrieval** - Agent uses the schema context we provided
3. **SQL Generation** - GPT-4o generates SQL based on the question and schema
4. **Tool Execution** - Agent calls `RunSqlTool` to execute the SQL
5. **Result Formatting** - Results are returned as UI components (DataFrameComponent)
6. **Response** - We extract and display the results

## Why Schema Context is Critical

Compare these two approaches:

### Without Schema Context:

In [25]:
# This might generate incorrect SQL (looking for 'price' instead of 'total_amount')
await ask_agent(
    "What is the total revenue from delivered orders?",
    include_schema=False  # No schema context
)

🤔 Question: What is the total revenue from delivered orders?

📊 Results:


,total_revenue
0,319480.59


### With Schema Context:

In [26]:
# This will generate correct SQL (using 'total_amount')
await ask_agent(
    "What is the total revenue from delivered orders?",
    include_schema=True  # Schema context included
)

🤔 Question: What is the total revenue from delivered orders?

📊 Results:


,total_revenue
0,319480.59


## Verifying Agent Results

Let's verify the Agent's answer with direct SQL:

In [27]:
# Direct SQL query for verification
verification_result = run_sql_simple("""
SELECT 
    COUNT(*) as delivered_count,
    SUM(total_amount) as total_revenue
FROM orders
WHERE status = 'Delivered'
""")

print("=== VERIFICATION (Direct SQL) ===")
display(verification_result)

=== VERIFICATION (Direct SQL) ===


/var/folders/xz/5_crg54x7fqbg35gh1v4rcbh0000gn/T/ipykernel_32684/4022728488.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,delivered_count,total_revenue
0,133,319480.59


---

# Section 8: Testing Various Question Types

## Simple Queries

In [28]:
await ask_agent("How many products are in stock?")

🤔 Question: How many products are in stock?

📊 Results:


,total_products_in_stock
0,13561


## Aggregation Queries

In [29]:
await ask_agent("What is the average product price?")

🤔 Question: What is the average product price?

📊 Results:


,average_price
0,1025.7912000000000000


In [30]:
await ask_agent("What is the highest order value?")

🤔 Question: What is the highest order value?

📊 Results:


,highest_order_value
0,4988.40


## Filtering and Sorting

In [31]:
await ask_agent("Show me the 10 most expensive products")

🤔 Question: Show me the 10 most expensive products

📊 Results:


,name,price
0,Bag,1996.52
1,"Analytics Platform Jones, Wolfe and Sanchez",1958.66
2,Adapter,1947.97
3,Router,1939.33
4,"CRM System Peterson, Weber and Shaw",1881.26
5,"CRM System Adams, Stout and Morales",1820.15
6,Charger Pollard-Williams,1805.79
7,Case,1720.09
8,Mouse Contreras-Castaneda,1710.98
9,Technical Manual,1709.89


In [32]:
await ask_agent("Which customers are from the USA?")

🤔 Question: Which customers are from the USA?

📊 Results:


,name,email
0,Jose Hayes,perezkimberly@example.com
1,David Miller,joneskimberly@example.com
2,Jeffrey Carey,orandall@example.org
3,Joseph Wilson,joseph42@example.net
4,Anita Green,daviddonovan@example.org
5,Nichole Hall,megan13@example.org
6,Larry Cook,troy47@example.org
7,David Lawrence,roachvanessa@example.com
8,Lauren Shaffer,erinreese@example.com


## Complex JOIN Queries

In [33]:
await ask_agent("How many orders has each customer segment placed?")

🤔 Question: How many orders has each customer segment placed?

📊 Results:


,segment,order_count
0,Individual,62
1,Enterprise,59
2,SMB,79


In [34]:
await ask_agent("Which customers have never placed an order?")

🤔 Question: Which customers have never placed an order?

📊 Results:


,id,name,email
0,7,Joshua Knight,dwalton@example.net
1,17,Marc Saunders,hector28@example.com
2,23,John Hull,alisonsellers@example.com
3,25,Kimberly Castillo,wilsonjennifer@example.net
4,39,Melissa Williams,james00@example.net
5,47,Courtney Graham,wendy56@example.net
6,48,Anita Green,daviddonovan@example.org
7,55,Angela Hicks,lrobbins@example.net
8,58,Blake James,kellison@example.com
9,59,Timothy Jones,ephillips@example.net


---

# Section 9: Error Handling and Best Practices

## Common Issues and Solutions

### Issue 1: Ambiguous Questions

**Bad:** "Show me the data" (What data? Which table?)

**Good:** "Show me all customers from the USA"

In [35]:
# This will likely generate better SQL
await ask_agent("Show me all customers from the USA")

🤔 Question: Show me all customers from the USA

📊 Results:


,id,name,email,segment,country,created_at,updated_at
0,2,Jose Hayes,perezkimberly@example.com,SMB,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
1,15,David Miller,joneskimberly@example.com,SMB,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
2,20,Jeffrey Carey,orandall@example.org,Individual,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
3,44,Joseph Wilson,joseph42@example.net,Enterprise,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
4,48,Anita Green,daviddonovan@example.org,Individual,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
5,68,Nichole Hall,megan13@example.org,Individual,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
6,74,Larry Cook,troy47@example.org,SMB,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
7,85,David Lawrence,roachvanessa@example.com,SMB,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798
8,90,Lauren Shaffer,erinreese@example.com,Individual,USA,2026-04-22 19:54:35.452798,2026-04-22 19:54:35.452798


### Issue 2: Incorrect Column Names

**Problem:** Agent looks for 'price' in orders table (doesn't exist)

**Solution:** Provide clear schema documentation (we do this in `SCHEMA_CONTEXT`)

In [36]:
# Schema context prevents this error
await ask_agent("What is the total revenue?")  # Will correctly use 'total_amount'

🤔 Question: What is the total revenue?

📊 Results:


,total_revenue
0,500684.33


### Issue 3: Case Sensitivity

**Problem:** PostgreSQL string comparisons are case-sensitive

**Solution:** Document exact values in schema context

In [37]:
# This works because we documented 'Delivered' (capitalized) in schema
await ask_agent("Show delivered orders")

🤔 Question: Show delivered orders

📊 Results:


,id,customer_name,order_date,total_amount,status,shipping_address
0,2,Brandi Martinez,2025-06-09,83.31,Delivered,"71392 Kelly Well\nFitzgeraldside, NM 77501"
1,3,Brandi Martinez,2025-12-18,2349.80,Delivered,Unit 5594 Box 6584\nDPO AA 94613
2,4,Larry Cook,2026-03-21,3058.13,Delivered,"3460 Fischer Flat\nNorth Christopher, WI 49313"
3,5,Brandi Smith,2026-02-15,1646.48,Delivered,"7750 Cathy Cape\nEast Mark, KS 08991"
4,7,Paul Woods,2025-10-30,1260.84,Delivered,"8204 Dawson Streets\nNorth Lauren, PR 79787"
...,...,...,...,...,...,...
128,194,Jeffrey Montgomery,2025-05-07,997.28,Delivered,None
129,195,Johnathan Mitchell,2026-02-15,4356.10,Delivered,"756 Parker Parks\nSouth Glenn, MH 58894"
130,196,Brittany Roy,2025-06-09,2415.24,Delivered,"8075 Henry Rest Suite 021\nLake Karenville, VI..."
131,198,Natalie Cisneros,2025-08-04,1862.27,Delivered,"596 James Centers\nNew Jonathanmouth, NC 65249"


## Best Practices

1. **Always provide schema context** - Critical for accuracy
2. **Be specific in questions** - Avoid ambiguous language
3. **Document exact values** - Include enum values, case sensitivity
4. **Verify results** - Compare Agent output with direct SQL
5. **Use descriptive column names** - Helps the LLM understand intent
6. **Include business logic** - Document relationships and constraints
7. **Test edge cases** - Empty results, NULLs, etc.
8. **Monitor and log** - Track query accuracy in production

## Production Considerations

1. **User Permissions** - Implement proper `UserResolver` with auth
2. **SQL Validation** - Block dangerous operations (DROP, DELETE without WHERE)
3. **Rate Limiting** - Prevent abuse
4. **Query Approval** - Show SQL to users before execution
5. **Monitoring** - Track failed queries for retraining
6. **Caching** - Cache common questions
7. **Row-Level Security** - Filter data based on user permissions
8. **API Key Management** - Never hardcode credentials

---

# Section 10: Next Steps and Resources

## What We've Learned

✅ Text-to-SQL fundamentals
✅ Vanna.ai 2.0 Agent architecture
✅ Setting up LLM, database, tools, and user resolver
✅ Importance of schema documentation
✅ Querying with natural language
✅ Handling async operations correctly
✅ Extracting results from UI components
✅ Best practices and error handling

## Key Takeaways

1. **Schema context is critical** - Without it, the Agent will generate incorrect SQL
2. **Vanna 2.0 uses async generators** - Use `async for` to iterate components
3. **UI components hold data in different attributes** - DataFrameComponent uses `rows`
4. **Direct SQL is useful** - For verification and simple queries
5. **Production requires more** - Permissions, validation, monitoring

## Advanced Topics to Explore

- **Custom Tools** - Create your own tools beyond RunSqlTool
- **Multi-Turn Conversations** - Agent memory and context
- **Different LLMs** - Try Claude, Llama, or other models
- **Different Databases** - MySQL, SQLite, Snowflake
- **Web Interface** - Vanna's `<vanna-chat>` component
- **Security** - Row-level security, query validation
- **Performance** - Caching, query optimization

## Resources

- **Vanna.ai Documentation**: https://docs.vanna.ai/
- **GitHub Repository**: https://github.com/vanna-ai/vanna
- **Examples**: https://github.com/vanna-ai/vanna/tree/main/examples
- **Community**: Discord, GitHub Discussions
- **API Reference**: https://docs.vanna.ai/api/

## Your Next Project Ideas

1. **Build a Slack Bot** - Answer data questions in Slack
2. **Create a Dashboard** - Visualize query results
3. **HR Analytics** - Query employee database
4. **Sales Intelligence** - Real-time sales queries
5. **Customer Support** - Automated data lookups

---

## Thank You!

You've completed the Vanna.ai Text-to-SQL tutorial. You now have the foundation to build your own Text-to-SQL applications!

**Happy coding! 🚀**